# Topic C3: Leakage & Split Guardian
### Cross-Split Leakage Detection, Severity Quantification & Graph-Stratified Re-Splitting

This notebook provides a complete, runnable reproduction of the **SplitGuardian** engine:
1. **Dataset Generation:** Generates multi-camera, video burst visual dataset with known Ground Truth.
2. **Dual-Leakage Demonstration:** Temporal burst near-duplicates and multi-camera viewpoint shifts.
3. **Primary Metric:** Leakage Detection F1 against baselines.
4. **Re-Splitting:** Naive random split vs Group-Preserving Stratified Re-Split.
5. **Comparative Model Evaluation:** Proving whether validation metrics are 'artificially inflated' (*đẹp giả*).

In [ ]:
# Step 1: Setup and Imports
import os
import sys
import json
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from data.dataset_generator import generate_benchmark_dataset
from src.detectors.hash_detector import HashDetector
from src.detectors.embedding_detector import EmbeddingDetector
from src.detectors.hybrid_detector import HybridLeakageDetector
from src.splitters.naive_splitter import NaiveRandomSplitter
from src.splitters.guardian_splitter import GuardianStratifiedSplitter
from src.evaluation.leakage_metrics import compute_pairwise_detection_metrics, compute_split_leakage_audit
from src.evaluation.model_trainer import train_and_evaluate_split

print('PyTorch Version:', torch.__version__)
print('All modules successfully imported!')

## 1. Benchmark Dataset Generation with Dual-Leakage Ground Truth
We generate a synthetic dataset with:
- 14 Train/Val scenes and 4 Independent Holdout Test scenes
- 4 classes (sedan, truck, motorcycle, emergency vehicle)
- 2 distinct forms of cross-split leakage: **Temporal Burst** and **Multi-Camera Viewpoint Overlap**
- Exhaustive ground truth manifest with pairwise labels.

In [ ]:
# Step 2: Generate dataset
dataset_dir = os.path.abspath('../dataset_store')
manifest_path, gt_pairs_path = generate_benchmark_dataset(
    output_dir=dataset_dir,
    num_trainval_scenes=14,
    num_test_scenes=4,
    burst_per_scene=3,
    frames_per_burst=4,
    num_cameras=2,
    seed=42
)

with open(manifest_path, 'r') as f:
    metadata = json.load(f)
with open(gt_pairs_path, 'r') as f:
    gt_pairs = json.load(f)['pairs']

trainval = [m for m in metadata if not m['is_holdout_scene']]
holdout = [m for m in metadata if m['is_holdout_scene']]
trainval_ids = set(m['sample_id'] for m in trainval)
trainval_gt = [p for p in gt_pairs if p['sample_a'] in trainval_ids and p['sample_b'] in trainval_ids]

print(f'Total samples: {len(metadata)}')
print(f'Train/Val pool: {len(trainval)} samples')
print(f'Holdout Test set: {len(holdout)} samples')
print(f'Ground Truth leak pairs in Train/Val pool: {len(trainval_gt)}')

# Load images
images_lookup = {m['sample_id']: Image.open(os.path.join(dataset_dir, m['file_path'])) for m in metadata}

## 2. Leakage Detection Methods & Primary Metric Evaluation
We evaluate three detection strategies on the Primary Metric (**Leakage Detection F1**):
1. **Baseline 1:** Perceptual Hash (dHash/pHash with Hamming distance threshold)
2. **Baseline 2:** Deep CNN Visual Embeddings (Cosine similarity threshold)
3. **Proposed:** SplitGuardian Hybrid Multimodal Graph Detector

In [ ]:
# Step 3: Run Detectors
# 3.1 Perceptual Hash Baseline
hash_det = HashDetector(hash_size=16, hamming_threshold=10)
hash_pairs, _ = hash_det.detect_pairwise_leakage(trainval, images_lookup)
hash_metrics = compute_pairwise_detection_metrics(trainval_gt, hash_pairs, len(trainval))

# 3.2 Deep Visual Embeddings
emb_det = EmbeddingDetector(sim_threshold=0.82)
embeddings = emb_det.compute_all_embeddings(metadata, images_lookup)
emb_pairs = emb_det.detect_pairwise_leakage(trainval, embeddings)
emb_metrics = compute_pairwise_detection_metrics(trainval_gt, emb_pairs, len(trainval))

# 3.3 SplitGuardian Hybrid Graph Detector
hybrid_det = HybridLeakageDetector()
hybrid_analysis = hybrid_det.analyze_dataset(trainval, images_lookup)
hybrid_metrics = compute_pairwise_detection_metrics(trainval_gt, hybrid_analysis['predicted_leak_pairs'], len(trainval))

print(f'Baseline pHash      -> Precision: {hash_metrics["precision"]:.4f} | Recall: {hash_metrics["recall"]:.4f} | F1: {hash_metrics["f1"]:.4f}')
print(f'Deep Embeddings     -> Precision: {emb_metrics["precision"]:.4f} | Recall: {emb_metrics["recall"]:.4f} | F1: {emb_metrics["f1"]:.4f}')
print(f'SplitGuardian Hybrid-> Precision: {hybrid_metrics["precision"]:.4f} | Recall: {hybrid_metrics["recall"]:.4f} | F1: {hybrid_metrics["f1"]:.4f}')

## 3. Naive Random Split vs SplitGuardian Re-Split Audit
We compare:
- **Naive Random Split:** Shuffles samples as i.i.d., causing burst & camera leakage.
- **SplitGuardian Re-Split:** Group-Preserving Stratified Split allocating whole atomic clusters.

In [ ]:
# Step 4: Execute Splitting Strategies
# 4.1 Naive Random Split
naive_splitter = NaiveRandomSplitter(train_ratio=0.75, val_ratio=0.25, seed=42)
leaky_train_ids, leaky_val_ids, _ = naive_splitter.split(trainval)

# 4.2 SplitGuardian Re-Split
guardian_splitter = GuardianStratifiedSplitter(target_train_ratio=0.75, target_val_ratio=0.25, target_test_ratio=0.0, seed=42)
clean_train_ids, clean_val_ids, _, tradeoff = guardian_splitter.split_by_clusters(trainval, hybrid_analysis['cluster_assignment'])

# Audit residual leakage
leaky_audit = compute_split_leakage_audit(leaky_train_ids, leaky_val_ids, trainval_gt, embeddings)
clean_audit = compute_split_leakage_audit(clean_train_ids, clean_val_ids, trainval_gt, embeddings)

print(f'Naive Random Split     -> Cross Leaked Pairs: {leaky_audit["cross_leaked_pair_count"]} | Max Cosine Sim: {leaky_audit["cross_split_similarity"]["max_cosine_sim"]:.4f}')
print(f'SplitGuardian Re-Split -> Cross Leaked Pairs: {clean_audit["cross_leaked_pair_count"]} | Max Cosine Sim: {clean_audit["cross_split_similarity"]["max_cosine_sim"]:.4f}')

## 4. Model Training & Proof of "Đẹp Giả" (Generalization Gap)
We train identical PyTorch classifiers on both splits and evaluate them on the **True Independent Holdout Test Set** (unseen scenes 14-17).

> **LƯU Ý / BẪY:** Metric thấp hơn sau clean split không tự động nghĩa là model tệ hơn; nhóm phải chứng minh evaluation nào đáng tin hơn.

In [ ]:
# Step 5: Comparative Model Evaluation
meta_map = {m['sample_id']: m for m in metadata}
holdout_ids = [m['sample_id'] for m in holdout]

eval_leaky = train_and_evaluate_split('Leaky Random Split', leaky_train_ids, leaky_val_ids, holdout_ids, embeddings, meta_map, epochs=45, seed=42)
eval_clean = train_and_evaluate_split('SplitGuardian Clean Split', clean_train_ids, clean_val_ids, holdout_ids, embeddings, meta_map, epochs=45, seed=42)

print('--- COMPARATIVE MODEL RESULTS ---')
print(f'Leaky Split:  Validation Acc = {eval_leaky["validation_accuracy"]}% | Holdout Test Acc = {eval_leaky["holdout_test_accuracy"]}% | Generalization Gap = {eval_leaky["generalization_gap_percent"]}% (ĐẸP GIẢ)')
print(f'Clean Split:  Validation Acc = {eval_clean["validation_accuracy"]}% | Holdout Test Acc = {eval_clean["holdout_test_accuracy"]}% | Generalization Gap = {eval_clean["generalization_gap_percent"]}% (ĐÁNG TIN)')

## 5. Production CI/CD Gate Verification
We can run the automated security gate directly from Python or CLI to verify splits before training.

In [ ]:
# Step 6: Test CI/CD Gate on Clean Split
from src.cli.split_guardian_gate import run_ci_gate

exit_code = run_ci_gate(
    manifest_path='../dataset_store/metadata.json',
    policy_path='../configs/guardian_policy.yaml',
    split_assignment_path='../reports/clean_split.json'
)
print('Gate Exit Code:', exit_code, '(0 = PASSED)')